# 🎮 A/B-тестирование обновления игры "Castle Rush"

## Контекст

В мобильной игре **«Castle Rush»** стандартная конверсия в платящего игрока (платеж в течение первого месяца после установки) составляет **10%** (рассчитано на основе данных о **500** новых игроках, использованных как ретроспективный бейзлайн).

Команда геймдизайна выпустила обновление, которое, по их гипотезе, должно **повысить долю платящих игроков до 11%** (абсолютный прирост на 1 п.п.).

**Вопрос для LTV-менеджера:** Стоит ли раскатывать обновление на весь трафик, или наблюдаемый рост — просто статистическая случайность при малом размере выборки?

---

## Задача

Спроектируйте **A/B-тест** для проверки этой гипотезы.

### Условия:
- Дневной входящий трафик составляет **100 новых игроков**.

### Требования:

1. **Дизайн эксперимента**  
   — размер групп, длительность, способ рандомизации, ключевые метрики.

2. **Минимальная длительность эксперимента** (в днях), достаточная для детектирования эффекта с заданной мощностью.

3. **Синтетический датасет** для **контрольной группы** (конверсия 10%)  
   — построить **доверительный интервал (ДИ)** для конверсии контрольной группы и прокомментировать результат.

4. **Три сценария для экспериментальной группы:**
   - 🔻 **Худший** — конверсия ниже контрольной (9%)
   - ➖ **Нулевой** — конверсия равна контрольной (10%)
   - 🔺 **Лучший** — конверсия соответствует гипотезе (11%)

5. Для каждого сценария рассчитать:
   - Фактическую разницу конверсий (эффект)
   - Доверительный интервал (ДИ) для разницы
   - **Вывод:** отвергаем нулевую гипотезу (\(H_0\)) или нет?

6. **Альтернативный подход:**  
   Провести **бутстреп** для оценки устойчивости выводов.

## Решение

**1. Дизайн эксперимента:**

*Гипотеза:* Обновление мобильной игры позволит увеличить конверсию в оплату на 1% для всех, кто установил игру.
*Что делаем:* Выпускаем обновление мобильной игры
*На ком тестируем:* на всех пользователях, которые установили игру
*Метрики:* ключевая метрика - конверсия в покупку, контрметрика - отток (не рассмотрена в данном задании)
*Ожидаемый эффект:* конверсия в покупку вырастет при неизменных показателях оттока
*План действий в зависимости от результатов эксперимента:* eсли наш эксперимент будет положительным и мы зафиксируем ожидаемое улучшение в ключевых метриках и не понизим контрметрику - мы оценим практическую значимость результата, в случае статистической и практической значимости - выпускаем обновление на всех, в случае, если изменений нет - оставляем как есть или внедряем, если обновление нужно для других целей.

In [37]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.stats.proportion as proportion
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.power import zt_ind_solve_power
from statsmodels.stats.proportion import proportion_effectsize

In [38]:
#рассчитываем длительность эксперимента
alpha = 0.05 #фиксируем ошибку 1 рода
beta = 0.8
power = 1 - beta #фиксируем мощность
p1 = 0.1 #доля плательщиков в первой группе
p2 = 0.11 #ожидаемая доля плательщиков во второй
MDE = 0.01 #минимально обнаруживаемый эффект
effect_size = proportion_effectsize(p1, p2) #подсчет размера эффекта
#считаем размер выборки с помощью библиотеки statsmodels.stats.power для Z-теста (z-тест используем, так как сравниваем средние, а дисперсия ГС
#можем выразить)
n = zt_ind_solve_power(effect_size=effect_size, alpha=alpha, power=power, ratio=1)
base_n = 500 #уже имеющиеся пользователи на платформе
days = round((n * 2 - base_n) / 100) #считаем длительность в днях с учетом того что не надо "ждать" 500 существующих
days

42

In [39]:
n_need = round(n) * 2 #необходимое количество человек для эксперимента в двух группах
n_need

4668

In [59]:
import random
np.random.seed(2026)
random.seed(2026)

In [68]:
#произведем АА-тест, чтобы понять что правильно делим пользователей
users = pd.DataFrame({'paying_user': np.random.binomial(1, 0.10, base_n)}) #берем выборку с существующими пользователями и проверяем, что мы их
#правильно делим, так как пользователей мало, для достижения эффекта - мы для контрольной и тестовой группы будем брать 50% пользователей случайным
#образом
def aa_test(data, num_simul=1000): #будем перемешивать пользователей и делить их пополам 1000 раз, если деление производим правильно - между группами
    #в 95% случаев не будет статистически значимой разницы
    false_positive = 0 #считаем количество ложных срабатываний теста
    for i in range(num_simul):
        shuffled = np.random.permutation(data) #перемешиваем
        half = len(shuffled) // 2
        f_half = shuffled[:half] #делим пополам
        s_half = shuffled[half:]
        z_value, p_value_z = proportion.proportions_ztest([sum(f_half), sum(s_half)], [len(f_half), len(s_half)])
        if p_value_z < alpha: #проверяем условие
            false_positive += 1
    print(f'Доля ложных срабатываний {round(false_positive / num_simul*100, 3)}%')

z = round(stats.norm.ppf(1 - alpha/2), 2)
num_simul = 1000
aa_test(users['paying_user'].values, num_simul)
# Ожидаемое число ложных срабатываний
expected_fp = num_simul * alpha 
std_fp = np.sqrt(num_simul * alpha * (1 - alpha))
lower_bound = expected_fp - z * std_fp
upper_bound = expected_fp + z * std_fp

print(f"Ожидаемое число ложных срабатываний: {expected_fp}")
print(f"95% доверительный интервал: {expected_fp} ± {z} × {std_fp:.2f} = [{lower_bound:.1f}, {upper_bound:.1f}]")
print(f"В процентах: [{lower_bound/10:.2f}%, {upper_bound/10:.2f}%]")

Доля ложных срабатываний 4.8%
Ожидаемое число ложных срабатываний: 50.0
95% доверительный интервал: 50.0 ± 1.96 × 6.89 = [36.5, 63.5]
В процентах: [3.65%, 6.35%]


Так как полученная доля ложных срабатываний находится внутри доверительного интервала, то деление аудитории правильное

In [69]:
#сгененрируем датасет c группой контроля
n_required = round(n)
print(n_required)
users = pd.DataFrame({'paying_user': np.random.binomial(1, 0.10, n_required)}) #задаем конверсию 10%, размер группы - необходимый по расчету для обнару
#жения эффекта
users

2334


,paying_user
0,0
1,0
2,0
3,0
4,0
...,...
2329,0
2330,0
2331,0
2332,0


In [70]:
n_c = len(users)
m_c = sum(users['paying_user'])
w_с = m_c/n_c
w_с

0.10068551842330763

In [71]:
U = stats.norm(0,1)
U_kr = U.ppf(1 - alpha/2)
left_dov = w_с - np.sqrt(w_с * (1 - w_с) / n_required) * U_kr
right_dov = w_с + np.sqrt(w_с * (1 - w_с) / n_required) * U_kr
print(f'Доверительный интервал: [{round(left_dov, 3)};{round(right_dov, 3)}]')

Доверительный интервал: [0.088;0.113]


In [72]:
#рассмотрим различные варианты возможных результатов эксперимента

#генерируем датасет с худшим показателем
users_test = pd.DataFrame({'paying_user': np.random.binomial(n=1, p=0.08, size=n_required)})

#делаем Z-тест
m_t_1 = sum(users_test['paying_user'])
n_t_1 = len(users_test)
z_value, p_value_z = proportion.proportions_ztest([m_c, m_t_1], [n_c, n_t_1])
if (p_value_z < alpha):
    print('отвергаем H_0')
else:
    print('принимаем H_0')

отвергаем H_0


In [73]:
proportion_effectsize(m_c/n_c, m_t_1/n_t_1)

np.float64(0.07499170495152563)

Различие статистически значимо, расчет эффекта показал положительное значение - значит, эффект отрицательный, но так как 
size_effect < 0.2, он достаточно маленький

In [74]:
w_t_1 = m_t_1/n_t_1
left_dov1 = w_t_1 - np.sqrt(w_t_1 * (1 - w_t_1) / n_required) * U_kr
right_dov1 = w_t_1 + np.sqrt(w_t_1 * (1 - w_t_1) / n_required) * U_kr
print(f'Доверительный интервал: [{round(left_dov1, 3)};{round(right_dov1, 3)}]')

Доверительный интервал: [0.068;0.09]


Реальная конверсия после внедрения c вероятностью 95% находится между 6.8% и 9%

In [75]:
w_diff1 = w_t_1 - w_с
left_diff1 = w_diff1 - np.sqrt(-w_diff1 * (1 + w_diff1) / n_required) * U_kr
right_diff1 = w_diff1 + np.sqrt(-w_diff1 * (1 + w_diff1) / n_required) * U_kr
print(f'Доверительный интервал: [{round(left_diff1, 3)};{round(right_diff1, 3)}]')

Доверительный интервал: [-0.027;-0.016]


Изменение конверсии может снизиться на величину от 1.6% до 2.7% - не стоит осуществлять внедрение. При стоимости подписки 300 рублей, потеря в  выручке составит от 480 р до 810 р в сутки, за год при ежедневном увеличении скачивающих игру на 100 человек, потеря составит от 175_200 до 295_650 р по сравнению с текущей версией.

In [80]:
#генерируем датасет с лучшим показателем
users_test = pd.DataFrame({'paying_user': np.random.binomial(n=1, p=0.1108, size=n_required)})

#делаем Z-тест
m_t_2 = sum(users_test['paying_user'])
n_t_2 = len(users_test)
z_value, p_value_z = proportion.proportions_ztest([m_c, m_t_2], [n_c, n_t_2])
if (p_value_z < alpha):
    print('отвергаем H_0')
else:
    print('принимаем H_0')

отвергаем H_0


In [81]:
proportion_effectsize(m_c/n_c, m_t_2/n_t_2)

np.float64(-0.07338166617253661)

Различие статистически значимо, расчет эффекта показал отрицательное значение - значит, эффект положительный, но так как size_effect < 0.2, он достаточно маленький

In [83]:
w_t_2 = m_t_2/n_t_2
left_dov2 = w_t_2 - np.sqrt(w_t_2 * (1 - w_t_2) / n_required) * U_kr
right_dov2 = w_t_2 + np.sqrt(w_t_2 * (1 - w_t_2) / n_required) * U_kr
print(f'Доверительный интервал: [{round(left_dov2, 3)};{round(right_dov2, 3)}]')

Доверительный интервал: [0.11;0.137]


In [84]:
#посчитаем дов. интервал с помощью бутстрапа
bootstrap = 5000
conv = []
for i in range(bootstrap):
    sample = np.random.choice(users_test['paying_user'], size=len(users_test['paying_user']), replace=True)
    conv.append(sample.mean())
print(np.percentile(conv, [2.5, 97.5]))

[0.11096829 0.13667524]


Практически нет разницы между доверительным интервалом, посчитанным по нормальному стандартному распределению и с помощью бутстрапа

Реальная конверсия после внедрения c вероятностью 95% находится между 10.3% и 12.9%

In [86]:
w_diff2 = w_t_2 - w_с
left_diff2 = w_diff2 - np.sqrt(w_diff2 * (1 - w_diff2) / n_required) * U_kr
right_diff2 = w_diff2 + np.sqrt(w_diff2 * (1 - w_diff2) / n_required) * U_kr
print(f'Доверительный интервал: [{round(left_diff2, 3)};{round(right_diff2, 3)}]')

Доверительный интервал: [0.017;0.029]


Изменение конверсии может увеличиться на величину от 1% до 2% - стоит осуществлять внедрение при неизменных показателях оттока и при условии, что стоимость разработки и внедрения обновления не превышает дополнительный рост выручки от внедрения - согласно условию, обновление уже разработано, тогда остается необходимость неизменного оттока. При стоимости подписки 300 рублей, дополнительный рост в выручке составит от 300р до 600 р в сутки, за год при ежедневном увеличении скачивающих игру на 100 человек, дополнительный рост составит составит от 109_500 до 219_000 р по сравнению с текущей версией.

In [87]:
#генерируем датасет с таким же показателем
users_test = pd.DataFrame({'paying_user': np.random.binomial(n=1, p=0.10, size=n_required)})

#делаем Z-тест
m_t_3 = sum(users_test['paying_user'])
n_t_3 = len(users_test)
z_value, p_value_z = proportion.proportions_ztest([m_c, m_t_3], [n_c, n_t_3])
if (p_value_z < alpha):
    print('отвергаем H_0')
else:
    print('принимаем H_0')

принимаем H_0


In [88]:
proportion_effectsize(m_c/n_c, m_t_3/n_t_3)

np.float64(-0.008495231476225329)

In [89]:
w_t_3 = m_t_3/n_t_3
left_dov3 = w_t_3 - np.sqrt(w_t_3 * (1 - w_t_3) / n) * U_kr
right_dov3 = w_t_3 + np.sqrt(w_t_3 * (1 - w_t_3) / n) * U_kr
print(f'Доверительный интервал: [{round(left_dov3, 3)};{round(right_dov3, 3)}]')

Доверительный интервал: [0.091;0.116]


In [90]:
w_diff3 = w_t_3 - w_с
left_diff3 = w_diff3 - np.sqrt(abs(w_diff3) * (1 - abs(w_diff3)) / n_required) * U_kr
right_diff3 = w_diff3 + np.sqrt(abs(w_diff3) * (1 - abs(w_diff3)) / n_required) * U_kr
print(f'Доверительный интервал: [{round(left_diff3, 3)};{round(right_diff3, 3)}]')

Доверительный интервал: [0.001;0.005]


Изменения конверсии либо не будет, либо она может увеличиться на величину 0.2%, эффект составляет 0.002 < 0.2 - очень маленький, внедрять изменение стоит, если оно решает какие-то другие проблемы (увеличивает производительность, предусматривает дальнейшее расширение функций итд), при условии, что в группе Б нет изменения оттока.